In [1]:
# 180吸附，带朝向功能，不带角度调节功能，可以同时生成多个位点

from ase.io import read, write
from ase import Atoms
import numpy as np
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import Delaunay, distance_matrix

# ============================================================
# User controls (edit here)
# ============================================================
slab_path = "/lustre/home/ucaqzh0/thermol/100_co2_absorption/POSCAR"
output_dir = "/lustre/home/ucaqzh0/thermol/100_co2_absorption/absorption_chemical_130"

# Switches for site generation
make_top = True
make_bridge = True
make_hollow = True

# Orientation vectors for each site (CO2 axis direction)
vec_top = [1, -1, 0]
vec_bridge = [1, -1, 0]
vec_hollow = [1, -1, 0]

# O-C-O bond angle (deg). 180 = linear, smaller = bent
oco_angle_top = 130
oco_angle_bridge = 130
oco_angle_hollow = 130

# Adsorption height and bond length
ads_height = 2.0
co_bond = 1.18


# ============================================================
# 1. Read slab
# ============================================================
slab = read(slab_path, format="vasp")
pos = slab.get_positions()
zpos = pos[:, 2]


# ============================================================
# 2. Layer detection by z clustering (robust)
# ============================================================

def cluster_layers(z_values, tol):
    z_sorted = np.sort(z_values)
    groups = [[z_sorted[0]]]
    for value in z_sorted[1:]:
        if abs(value - groups[-1][-1]) <= tol:
            groups[-1].append(value)
        else:
            groups.append([value])
    centers = np.array([np.mean(g) for g in groups])
    return centers, groups

z_sorted = np.sort(zpos)
diffs = np.diff(z_sorted)
diffs = diffs[diffs > 1e-3]
if len(diffs) > 0:
    median_diff = np.median(diffs)
    layer_tol = min(0.30, median_diff * 0.5)
else:
    layer_tol = 0.30

centers, _groups = cluster_layers(zpos, tol=layer_tol)
centers = np.sort(centers)

surface_z = centers.max()
print(f"Detected {len(centers)} layers (tol={layer_tol:.3f} Å). Surface z = {surface_z:.3f} Å")


# ============================================================
# 3. Extract atoms belonging to the top layer
# ============================================================
layer_cut = max(0.15, layer_tol)
surf_idx = np.where(abs(zpos - surface_z) < layer_cut)[0]
surf_xy = pos[surf_idx][:, :2]

print(f"Surface atoms: {len(surf_idx)}")


# ============================================================
# 4. Detect surface lattice type (111 / 100 / 110)
# ============================================================
cell = slab.get_cell()
a_vec = cell[0]
b_vec = cell[1]
a_len = np.linalg.norm(a_vec)
b_len = np.linalg.norm(b_vec)
angle = np.degrees(
    np.arccos(np.clip(np.dot(a_vec, b_vec) / (a_len * b_len), -1.0, 1.0))
)

face_type = None
if abs(angle - 60.0) < 5.0 and abs(a_len - b_len) / a_len < 0.05:
    face_type = "111"
elif abs(angle - 90.0) < 5.0 and abs(a_len - b_len) / a_len < 0.05:
    face_type = "100"
elif abs(angle - 90.0) < 5.0:
    face_type = "110"

if face_type is None:
    dist_mat = distance_matrix(surf_xy, surf_xy)
    np.fill_diagonal(dist_mat, 1e9)
    d_nn = np.min(dist_mat, axis=1).mean()
    second_nn = np.sort(dist_mat, axis=1)[:, 1].mean()
    ratio = second_nn / d_nn

    if abs(ratio - np.sqrt(3)) < 0.1:
        face_type = "111"
    elif abs(ratio - np.sqrt(2)) < 0.1:
        face_type = "100"
    else:
        face_type = "110"

print(f"Surface identified as Cu({face_type})")


# ============================================================
# 5. Stacking detection (ABC / AB / AAAAA…)
# ============================================================

def normalize_layer_pattern(xy):
    center = xy.mean(axis=0)
    xy_centered = xy - center
    return np.round(xy_centered, 3)


def layers_match(xy1, xy2, tol=0.2):
    a = normalize_layer_pattern(xy1)
    b = normalize_layer_pattern(xy2)
    nbrs = NearestNeighbors(n_neighbors=1).fit(b)
    distances, _ = nbrs.kneighbors(a)
    return np.all(distances < tol)


def detect_stacking_pattern(all_layers_xy):
    patterns = []
    labels = []
    next_label = 0
    for xy in all_layers_xy:
        matched = False
        for i, ref in enumerate(patterns):
            if layers_match(xy, ref):
                labels.append(chr(ord('A') + i))
                matched = True
                break
        if not matched:
            patterns.append(xy.copy())
            labels.append(chr(ord('A') + next_label))
            next_label += 1
    return "".join(labels)

# get all layers’ XY patterns (using z clustering)
all_layers_xy = []
for zc in centers:
    idx = np.where(abs(zpos - zc) < layer_cut)[0]
    layer_xy = pos[idx][:, :2]
    all_layers_xy.append(layer_xy)

stacking = detect_stacking_pattern(all_layers_xy)
print(f"Stacking sequence: {stacking}")


# ============================================================
# 6. Generate adsorption sites: top / bridge / hollow
# ============================================================
# ---- top ----
top_xy = surf_xy[0]

# ---- bridge ----
dist_mat = distance_matrix(surf_xy, surf_xy)
np.fill_diagonal(dist_mat, 1e9)
i, j = np.unravel_index(np.argmin(dist_mat), dist_mat.shape)
bridge_xy = (surf_xy[i] + surf_xy[j]) / 2

# ---- hollow (Delaunay circumcenters) ----

def triangle_circumcenter(a, b, c):
    ax, ay = a
    bx, by = b
    cx, cy = c
    d = 2 * (ax * (by - cy) + bx * (cy - ay) + cx * (ay - by))
    if abs(d) < 1e-10:
        return None
    ux = ((ax**2 + ay**2) * (by - cy) + (bx**2 + by**2) * (cy - ay) + (cx**2 + cy**2) * (ay - by)) / d
    uy = ((ax**2 + ay**2) * (cx - bx) + (bx**2 + by**2) * (ax - cx) + (cx**2 + cy**2) * (bx - ax)) / d
    return np.array([ux, uy])


def point_in_triangle(p, a, b, c, tol=1e-8):
    v0 = c - a
    v1 = b - a
    v2 = p - a
    dot00 = np.dot(v0, v0)
    dot01 = np.dot(v0, v1)
    dot02 = np.dot(v0, v2)
    dot11 = np.dot(v1, v1)
    dot12 = np.dot(v1, v2)
    inv_denom = 1 / (dot00 * dot11 - dot01 * dot01 + 1e-12)
    u = (dot11 * dot02 - dot01 * dot12) * inv_denom
    v = (dot00 * dot12 - dot01 * dot02) * inv_denom
    return (u >= -tol) and (v >= -tol) and (u + v <= 1 + tol)


tri = Delaunay(surf_xy)
hollows = []
for simplex in tri.simplices:
    pts = surf_xy[simplex]
    cc = triangle_circumcenter(pts[0], pts[1], pts[2])
    if cc is None:
        continue
    if point_in_triangle(cc, pts[0], pts[1], pts[2]):
        hollows.append(cc)

if not hollows:
    # fallback to triangle centroids
    for simplex in tri.simplices:
        pts = surf_xy[simplex]
        hollows.append(pts.mean(axis=0))

surface_center = surf_xy.mean(axis=0)
hollow_xy = min(hollows, key=lambda h: np.linalg.norm(h - surface_center))

print("Adsorption sites generated: top / bridge / hollow")


# ============================================================
# 7. Build CO2 and write POSCARs
# ============================================================
import numpy as np


def build_co2(angle_deg, bond_len):
    # Build CO2 with given O-C-O angle (deg). 180 = linear.
    if angle_deg >= 179.9:
        return Atoms('OCO', positions=[(-bond_len, 0, 0), (0, 0, 0), (bond_len, 0, 0)])
    phi = np.radians(angle_deg / 2.0)
    x = bond_len * np.cos(phi)
    y = bond_len * np.sin(phi)
    return Atoms('OCO', positions=[(x, y, 0), (0, 0, 0), (x, -y, 0)])


def align_co2_to_vector(mol, target_vec):
    # Rotate CO2 (bisector initially along x-axis) to target_vec direction
    v = np.array(target_vec, dtype=float)
    v /= np.linalg.norm(v)
    x = np.array([1.0, 0.0, 0.0])
    axis = np.cross(x, v)
    norm = np.linalg.norm(axis)
    if norm < 1e-12:
        # Parallel or anti-parallel
        if np.dot(x, v) > 0:
            return mol.copy()
        mol = mol.copy()
        mol.rotate(180, "y", center=mol.get_center_of_mass())
        return mol
    axis /= norm
    angle = np.degrees(np.arccos(np.clip(np.dot(x, v), -1.0, 1.0)))
    mol = mol.copy()
    mol.rotate(angle, axis, center=mol.get_center_of_mass())
    return mol


def write_ads_custom(xy, name, target_vec, oco_angle):
    mol = build_co2(oco_angle, co_bond)
    mol = align_co2_to_vector(mol, target_vec)
    mol.translate([xy[0], xy[1], surface_z + ads_height])
    system = slab + mol
    if slab.constraints:
        system.set_constraint(slab.constraints)
    out_path = f"{output_dir}/POSCAR_CO2_{name}.vasp"
    write(out_path, system, vasp5=True)
    print(f"  -> Written {out_path}")

import os
os.makedirs(output_dir, exist_ok=True)

if make_top:
    write_ads_custom(top_xy, "top_custom", vec_top, oco_angle_top)
if make_bridge:
    write_ads_custom(bridge_xy, "bridge_custom", vec_bridge, oco_angle_bridge)
if make_hollow:
    write_ads_custom(hollow_xy, "hollow_custom", vec_hollow, oco_angle_hollow)

print("All requested adsorption structures generated.")


Detected 6 layers (tol=0.300 Å). Surface z = 8.800 Å
Surface atoms: 8
Surface identified as Cu(100)
Stacking sequence: ABABAB
Adsorption sites generated: top / bridge / hollow
  -> Written /lustre/home/ucaqzh0/thermol/100_co2_absorption/absorption_chemical_130/POSCAR_CO2_top_custom.vasp
  -> Written /lustre/home/ucaqzh0/thermol/100_co2_absorption/absorption_chemical_130/POSCAR_CO2_bridge_custom.vasp
  -> Written /lustre/home/ucaqzh0/thermol/100_co2_absorption/absorption_chemical_130/POSCAR_CO2_hollow_custom.vasp
All requested adsorption structures generated.
